In [5]:
import sys
print(sys.executable)

/home/sam/my-study/agentic-ai/multimodal-rag-application/server/.venv/bin/python3


In [6]:
# Select the chat model
import os 
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

model = ChatOpenAI(model="gpt-5-nano")


### 1. Define Event Tools

In [53]:
from langchain.tools import tool

@tool
def create_calendar_event(
    title : str,
    start_time: str, # ISO format: "2024-01-15T14:00:00"
    end_time: str,    # ISO format: "2024-01-15T15:00:00"
    attendees: list[str],# email addresses
    location: str ="" 
)-> str:

    """Create a calendar event. Requires exact ISO datetime format"""
    # Stub: In practice, this would call Google Calendar API, Outlook API, etc.
    return f"Event created: {title} from {start_time} to {end_time} with {len(attendees)} attendees"

@tool
def send_email(
    to: list[str], # email addresses
    subject: str,
    body: str,
    cc: list[str] = []
)-> str:
    """Send an email via email API. Requires properly formatted addresses."""
    # Stub: In practice, this would call SendGrid, Gmail API, etc.
    return f"Email sent to {', '.join(to)} - Subject: {subject}"

@tool
def get_available_time_slots(
    attendees: list[str],
    date: str,  # ISO format: "2024-01-15"
    duration_minutes: int
) -> list[str]:
    """Check calendar availability for given attendees on a specific date."""
    # Stub: In practice, this would query calendar APIs
    return ["09:00", "14:00", "16:00"]


In [8]:
! uv pip show langchain

Using Python 3.11.14 environment at: /home/sam/my-study/agentic-ai/multimodal-rag-application/server/.venv
Name: langchain
Version: 1.2.18
Location: /home/sam/my-study/agentic-ai/multimodal-rag-application/server/.venv/lib/python3.11/site-packages
Requires: langchain-core, langgraph, pydantic
Required-by:


2. Create specialized sub agents 

Create a calendar agent
  - This agent understands natural language scheduling requests and translates them into precise API calls. It handles date parsing, availability checking and event creation

In [42]:
from langchain.agents import create_agent

CALENDAR_AGENT_PROMPT = (
    "You are calendar scheduling assistant."
    "***You MUST always call tools first to complete requests***."
    "Parse natural language scheuling requests(e.g. 'next Tuesday at 2pm') into proper ISO datetime formats."
    "Use get_available_time_slots to check availability when needed."
    "If there is no suitable time slot, stop and confirm unavailability in your response."
    "Use create_calendar_event to schedule events."
    "Always confirm what was scheduled in your final response."
)

calendar_agent = create_agent(
    model,
    tools = [create_calendar_event, get_available_time_slots],
    system_prompt = CALENDAR_AGENT_PROMPT
)

In [28]:
query = "Schedule a meeting next friday at 3 pm for 1 hr"
result = calendar_agent.invoke({"messages" : query })
result

{'messages': [HumanMessage(content='Schedule a meeting next friday at 3 pm for 1 hr', additional_kwargs={}, response_metadata={}, id='21b0d265-124b-46a3-a53c-93b328ad2845'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1579, 'prompt_tokens': 285, 'total_tokens': 1864, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1536, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DnJF2pnroUFqptv7mztTuAQNGGMC7', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e96ad-6b64-7a00-9b8c-3da2ede82c7d-0', tool_calls=[{'name': 'get_available_time_slots', 'args': {'attendees': [], 'date': '2026-06-12', 'duration_minutes': 60}, 'id': 'call_SEufJzB1wvVx6cYvcQpeftP8', 'type': 'tool_ca

In [17]:
query = "Schedule a meeting next friday at 2 pm for 1 hr"
result = calendar_agent.invoke({"messages" : query })
result

{'messages': [HumanMessage(content='Schedule a meeting next friday at 2 pm for 1 hr', additional_kwargs={}, response_metadata={}, id='a8b4b663-53b4-4d7d-a450-2716204f93f4'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1959, 'prompt_tokens': 284, 'total_tokens': 2243, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1920, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DnIsjX8XZ7NcPtpYYX8uMKW62DUhI', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e9698-5034-7263-abeb-0e96fb90beab-0', tool_calls=[{'name': 'get_available_time_slots', 'args': {'attendees': [], 'date': '2026-06-12', 'duration_minutes': 60}, 'id': 'call_Ms4k6MXVjUv08kdXf1jFpkgZ', 'type': 'tool_ca

In [32]:
query = "Schedule a meeting next Tuesday at 2pm for 1 hour"

events = calendar_agent.stream(input={"messages" : query },stream_mode="messages")

for event in events:
    print(event)

(AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai'}, id='lc_run--019e96b1-5fe9-72d1-8ae8-0fdd32f71e42', tool_calls=[{'name': 'get_available_time_slots', 'args': {}, 'id': 'call_vmTcFwggrBumHmTilYQQlvYW', 'type': 'tool_call'}], invalid_tool_calls=[], tool_call_chunks=[{'name': 'get_available_time_slots', 'args': '', 'id': 'call_vmTcFwggrBumHmTilYQQlvYW', 'index': 0, 'type': 'tool_call_chunk'}]), {'ls_integration': 'langchain_chat_model', 'langgraph_step': 1, 'langgraph_node': 'model', 'langgraph_triggers': ('branch:to:model',), 'langgraph_path': ('__pregel_pull', 'model'), 'langgraph_checkpoint_ns': 'model:770c232b-5c4a-a8ae-4b7a-4d93c7138326', 'checkpoint_ns': 'model:770c232b-5c4a-a8ae-4b7a-4d93c7138326', 'ls_provider': 'openai', 'ls_model_name': 'gpt-5-nano', 'ls_model_type': 'chat', 'ls_temperature': None})
(AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai'}, id='lc_run--019e96b1-5fe9-72d1-8ae8-

In [35]:
query = "Schedule a meeting next Tuesday at 2pm for 1 hour"

for step in calendar_agent.stream(input={"messages" : query },stream_mode="messages"):
    print (step)



(AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai'}, id='lc_run--019e96b3-c50c-7b11-90aa-213a5b02575f', tool_calls=[{'name': 'get_available_time_slots', 'args': {}, 'id': 'call_H29FVNxcwIoFfqYTrHqF2gfZ', 'type': 'tool_call'}], invalid_tool_calls=[], tool_call_chunks=[{'name': 'get_available_time_slots', 'args': '', 'id': 'call_H29FVNxcwIoFfqYTrHqF2gfZ', 'index': 0, 'type': 'tool_call_chunk'}]), {'ls_integration': 'langchain_chat_model', 'langgraph_step': 1, 'langgraph_node': 'model', 'langgraph_triggers': ('branch:to:model',), 'langgraph_path': ('__pregel_pull', 'model'), 'langgraph_checkpoint_ns': 'model:c1deaa42-8087-d2d7-15a8-7da1e7c4b050', 'checkpoint_ns': 'model:c1deaa42-8087-d2d7-15a8-7da1e7c4b050', 'ls_provider': 'openai', 'ls_model_name': 'gpt-5-nano', 'ls_model_type': 'chat', 'ls_temperature': None})
(AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai'}, id='lc_run--019e96b3-c50c-7b11-90aa-

In [40]:
query = "Schedule a meeting next Tuesday at 2pm for 1 hour"

for step in calendar_agent.stream(input={"messages" : query }):
    print (step.values()) # get only values

dict_values([{'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 684, 'prompt_tokens': 290, 'total_tokens': 974, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 640, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DnJPmvNr5vkCF3HY8GcZq5jcFt3rh', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e96b7-930e-7f93-888f-c89e7267c519-0', tool_calls=[{'name': 'get_available_time_slots', 'args': {'attendees': [], 'date': '2026-06-09', 'duration_minutes': 60}, 'id': 'call_tfAp0CWWT4VHjPKNjlfKD3MJ', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 290, 'output_tokens': 684, 'total_tokens': 974, 'input_token_details': {'audio': 0, 'cache

In [43]:
query = "Schedule a meeting next Tuesday at 2pm for 1 hour"

for step in calendar_agent.stream(input={"messages" : query }):
    print (step)

{'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1192, 'prompt_tokens': 290, 'total_tokens': 1482, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1152, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DnJQlpaxaByseFnVi79903rHB7LbG', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e96b8-8384-75b1-b5c8-dcb5340275fc-0', tool_calls=[{'name': 'get_available_time_slots', 'args': {'attendees': [''], 'date': '2026-06-09', 'duration_minutes': 60}, 'id': 'call_fXHgnLwGofiun97X3PY7tAkR', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 290, 'output_tokens': 1192, 'total_tokens': 1482, 'input_token_details': {'audio': 0, 'c

In [46]:
query = "Schedule a meeting next Tuesday at 2pm for 1 hour"

for step in calendar_agent.stream(input={"messages" : query }):

    for update in step.values(): # get only values
         for message in update.get("messages", []):
            message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  get_available_time_slots (call_P5CmEZw6e88FLXg3InPtxPuy)
 Call ID: call_P5CmEZw6e88FLXg3InPtxPuy
  Args:
    attendees: ['user']
    date: 2026-06-09
    duration_minutes: 60
================================= Tool Message =================================
Name: get_available_time_slots

["09:00", "14:00", "16:00"]
================================== Ai Message ==================================
Tool Calls:
  create_calendar_event (call_72dYtVZbhbCiGqGagVJGftOz)
 Call ID: call_72dYtVZbhbCiGqGagVJGftOz
  Args:
    title: Meeting
    start_time: 2026-06-09T14:00:00
    end_time: 2026-06-09T15:00:00
    attendees: ['user']
================================= Tool Message =================================
Name: create_calendar_event

Event created: Meeting from 2026-06-09T14:00:00 to 2026-06-09T15:00:00 with 1 attendees
================================== Ai Message ==================================


#### Step 3: Create email agent

In [56]:
EMAIL_AGENT_PROMPT = (
    "You are an email assistant. "
    "You MUST call send_email immediately for any send request."
    "If recipient email is unknown, use 'design-team@company.com' as default. "
    "If no sender is specified, use 'assistant@company.com'. "
    "Compose professional emails based on natural language requests. "
    "Extract recipient information and craft appropriate subject lines and body text. "
    "Use send_email to send the message. "
    "Always confirm what was sent in your final response."
)

email_agent = create_agent(
    model,
    tools=[send_email],
    system_prompt=EMAIL_AGENT_PROMPT,
)

In [57]:
query = "Send the design team a reminder about reviewing the new mockups"

for step in email_agent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  send_email (call_emnHjOgC37pLbon3FCiNattX)
 Call ID: call_emnHjOgC37pLbon3FCiNattX
  Args:
    to: ['design-team@company.com']
    subject: Reminder: Please review the latest mockups
    body: Hello Design Team,

This is a friendly reminder to review the latest mockups for the current project. Please share your feedback, questions, or concerns at your earliest convenience. The mockups are available in the Design Portal under the Latest Mockups section. If you would prefer a specific deadline, please let me know and I can set one.

Thank you,
Assistant
================================= Tool Message =================================
Name: send_email

Email sent to design-team@company.com - Subject: Reminder: Please review the latest mockups
================================== Ai Message ==================================

Email sent.

Details:
- To: design-team@company.com
- Subject: Reminder: P

#### Step 4: Wrap sub-agents as tools

Now add these worker agents as tools to the supervisor agent

In [71]:
@tool
def schedule_event(query: str) -> str:
    """Schedule calendar events using natural language.
    Use this when the user wants to create, modify, or check calendar appointment.
    Handles date/time parsing, availability checking and event creation.

    Input: Natural language scheduling request (e.g., 'meeting with design team next Tuesday at 2pm')
    """

    result = calendar_agent.invoke({
        "messages": [{"role": "user", "content": query}]
    })
    return result['messages'][-1].content

@tool
def manage_email(request: str) -> str:
    """Send emails using natural language.

    Use this when the user wants to send notifications, reminders, or any email
    communication. Handles recipient extraction, subject generation, and email
    composition.

    Input: Natural language email request (e.g., 'send them a reminder about
    the meeting')
    """
    result = email_agent.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    return result["messages"][-1].content

schedule_event("create a calendar next monday at 9 am")

In [68]:
schedule_event.invoke("create a calendar next monday at 9 am")

'Scheduled: 9:00 AM–10:00 AM on Monday, June 8, 2026\n- Title: 9:00 AM Meeting\n- Attendees: none\n- Location: none\n\nWould you like me to add attendees, a location, or reminders?'

### Step 5: Create Supervisor Agent

In [72]:
SUPERVISOR_PROMPT = (
    "You are a helpful personal assistant. "
    "You can schedule calendar events and send emails. "
    "Break down user requests into appropriate tool calls and coordinate the results. "
    "When a request involves multiple actions, use multiple tools in sequence."
)

supervisor_agent = create_agent(
    model,
    tools=[schedule_event, manage_email],
    system_prompt=SUPERVISOR_PROMPT,
)


In [73]:
query = "Schedule a team standup for tomorrow at 9am"

for step in supervisor_agent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  schedule_event (call_Lat2XEl6jh2m9zgACxqbJnxk)
 Call ID: call_Lat2XEl6jh2m9zgACxqbJnxk
  Args:
    query: Team standup tomorrow at 9am
================================= Tool Message =================================
Name: schedule_event

All set. I scheduled:

- Title: Team standup
- When: Tuesday, 2026-06-06, 09:00–09:15 (local time)
- Attendees: none specified
- Location: not set

Would you like me to add attendees (emails), set a location, or adjust the time or duration?
================================== Ai Message ==================================

All set. I scheduled a Team standup for tomorrow at 9:00–9:15 local time (date: 2026-06-06).

Attendees: none specified
Location: not set

Would you like me to:
- Invite the team (please provide attendee emails or a distribution list)?
- Add a location or a conferencing link (e.g., Zoom/Meet)?
- Set a reminder (e.g., 10 minutes before) or mak

In [74]:
query = (
    "Schedule a meeting with the design team next Tuesday at 2pm for 1 hour, "
    "and send them an email reminder about reviewing the new mockups."
)

for step in supervisor_agent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  schedule_event (call_AzQ2DRTlwkmTSrrQDo4k16sc)
 Call ID: call_AzQ2DRTlwkmTSrrQDo4k16sc
  Args:
    query: meeting with the design team next Tuesday at 2pm for 1 hour
  manage_email (call_y8RFmaBIoourI6G1jQUlCEYH)
 Call ID: call_y8RFmaBIoourI6G1jQUlCEYH
  Args:
    request: Send the design team a reminder email to review the new mockups.
================================= Tool Message =================================
Name: manage_email

Email sent.

Details:
- To: design-team@company.com
- Subject: Reminder: Please review the new mockups
- Body:
Hello Design Team,

This is a friendly reminder to review the latest mockups for the project. Please share any feedback or questions by replying to this email. The files are available in the project folder in our shared drive. If you need access, let me know.

Best regards,
Assistant

Sender: assistant@company.com

Would you like me to schedule a follo